In [1]:
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath(".."))
from google_play_scraper import reviews,app, Sort
from src.datapreprocessing import remove_duplicates,handle_missing_values,normalize_dates,save_dataset,rename_columns

# Web Scraping

In [2]:
# Define target apps
banks = [
    {
        "bank_name": "Commercial Bank of Ethiopia",
        "app_id": "com.combanketh.mobilebanking"
    },
    {
        "bank_name": "Bank of Abyssinia",
        "app_id": "com.boa.boaMobileBanking"
    },
    {
        "bank_name": "Dashen Bank",
        "app_id": "com.cr2.amolelight"
    }
]

all_reviews = []

TARGET_REVIEWS = 400

for bank in banks:
    print(bank)
    print(f"Collecting reviews for {bank['bank_name']}...")

    try:
        result, continuation_token = reviews(
            bank["app_id"],
            lang='en',
            country='et',
            sort=Sort.NEWEST,
            count=TARGET_REVIEWS
        )

        for review in result:
            all_reviews.append({
                "review_id": review.get("reviewId"),
                "review_text": review.get("content"),
                "rating": review.get("score"),
                "review_date": review.get("at"),
                "bank_name": bank["bank_name"],
                "source": "Google Play"
            })

        print(f"Collected {len(result)} reviews")

        # If fewer reviews returned
        if len(result) < TARGET_REVIEWS:
            print(
                f"Warning: Only {len(result)} reviews available "
                f"for {bank['bank_name']}."
            )

    except Exception as e:
        print(f"Error collecting reviews for {bank['bank_name']}: {e}")

{'bank_name': 'Commercial Bank of Ethiopia', 'app_id': 'com.combanketh.mobilebanking'}
Collected 400 reviews
{'bank_name': 'Bank of Abyssinia', 'app_id': 'com.boa.boaMobileBanking'}
Collected 400 reviews
{'bank_name': 'Dashen Bank', 'app_id': 'com.cr2.amolelight'}
Collected 400 reviews


In [3]:
# Create DataFrame
df = pd.DataFrame(all_reviews)

# Save to CSV
df.to_csv(r"../data/raw/bank_reviews.csv", index=False)

print(f"\nTotal reviews collected: {len(df)}")
print("Saved to bank_reviews_google_play.csv")
df.head()


Total reviews collected: 1200
Saved to bank_reviews_google_play.csv


,review_id,review_text,rating,review_date,bank_name,source
0,06f6640c-b65c-43e4-88ef-0a79be8b9534,it's a good application,5,2026-05-13 20:28:58,Commercial Bank of Ethiopia,Google Play
1,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,Commercial Bank of Ethiopia,Google Play
2,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,Commercial Bank of Ethiopia,Google Play
3,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13 12:19:17,Commercial Bank of Ethiopia,Google Play
4,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13 11:27:52,Commercial Bank of Ethiopia,Google Play


# Data Preprocessing

In [4]:
# Step 2: Remove Duplicates
df = remove_duplicates(df)

    # Step 3: Handle Missing Values
df = handle_missing_values(df)

    # Step 4: Normalize Dates
df = normalize_dates(df)

    # Step 5: Rename Columns
df = rename_columns(df)

    # Step 6: Select Final Columns
df = df[
        ["review_id", "review", "rating", "date", "bank", "source"]
    ]
print(os.getcwd())
    # Step 7: Save Cleaned Dataset
save_dataset(df,"cleaned_bank_reviews.csv")

Removed 0 duplicate reviews.
Removed 0 rows with missing values.
c:\Users\bemnet\Desktop\10academy\week2\fintech-review-analytics\fintech-review-analytics\notebooks
cleaned_bank_reviews.csv

Dataset saved as cleaned_bank_reviews.csv
